# Day 1 · Afternoon exercises (Exercise B)

**13:30–15:00 · then everyone back in the room at 15:00 for Peer review B · daily diary on Mentimeter at 15:45**

Bayesian statistics for PhD researchers, Aalborg University 2026. This continues the morning notebook. Summaries use dplyr and figures use ggplot2, as this morning.

## Plan for this session

| Part | What | Guide time |
|---|---|---|
| — | Rotate roles; load your group’s data | 5 min |
| B1 | Pooled and individual views | 10 min |
| B2 | Who is consistently shorter or longer? | 15 min |
| B3 | Does feedback improve accuracy, for everybody? | 15 min |
| B4 | One plot of every person’s trials | 5 min |
| B5 | Predict your own next attempt | 10 min |
| B6 | Density versus probability | 10 min |
| B7 | Your group’s question set | 15 min |

**Core:** B1–B6 and the question set. The optional extensions at the end are for spare time only.

**Rotate roles:** new coder, checker, two question writers, interviewer and reporter.

## Load your group’s data

Put **all** group members’ CSV files in the two folders: no-feedback files in `data/no_feedback/`, feedback files in `data/feedback/` (the file names are the same in both, for example `G3_P07.csv`). Then run the cell. Without files, it uses the synthetic backup group G0.

The count underneath should show n = 30 for each person in each condition.

In [ ]:
options(width = 90, repr.plot.width = 9, repr.plot.height = 6, repr.plot.bg = "white")
# Webpage exports go in two folders next to this notebook:
#   data/no_feedback/  files from collection/1_time_game.html                (e.g. G3_P07.csv)
#   data/feedback/     files from collection/2_time_game_with_feedback.html  (same file name!)
read_folder <- function(folder, condition) {
  files <- list.files(folder, pattern = "^G([1-9]|10)_P[0-9]+[.]csv$", full.names = TRUE)
  skipped <- setdiff(list.files(folder, pattern = "[.]csv$"), basename(files))
  if (length(skipped)) warning("Ignored in ", folder, ": ", paste(skipped, collapse = ", "),
                               ". Rename to the G3_P07.csv form to include it.", call. = FALSE)
  if (length(files) == 0) return(NULL)
  parts <- lapply(files, function(f) {
    x <- read.csv(f, stringsAsFactors = FALSE)
    if (!all(x$feedback_condition == condition))
      stop(basename(f), " is a '", x$feedback_condition[1], "' file but is in ", folder,
           ". Move it to the right folder.", call. = FALSE)
    if (nrow(x) != 30) warning(basename(f), " has ", nrow(x), " rows, not 30. Was the session finished?", call. = FALSE)
    x
  })
  do.call(rbind, parts)
}
# Synthetic backup: six fictional people, used only when both folders are empty.
make_backup <- function() {
  set.seed(2026)
  people <- data.frame(participant_id = sprintf("P%02d", 1:6),
                       bias_nf = c(-0.15, -0.08, 0.00, 0.05, 0.12, -0.02),
                       bias_fb = c(-0.04, -0.03, 0.01, 0.02, 0.03, -0.07),
                       noise   = c(0.06, 0.10, 0.08, 0.07, 0.09, 0.12))
  rows <- list()
  for (cond in c("no_feedback", "feedback")) for (i in seq_len(nrow(people))) for (target in c(5, 10, 15)) {
    p <- people[i, ]
    bias <- if (cond == "feedback") p$bias_fb else p$bias_nf
    elapsed <- round(target * (1 + bias + rnorm(10, 0, p$noise)), 3)
    rows[[length(rows) + 1]] <- data.frame(
      group_id = "G0", participant_id = p$participant_id, group = "G0",
      participant = paste0("G0", p$participant_id), trial = 1:10, target_seconds = target,
      elapsed_seconds = elapsed, elapsed_milliseconds = round(elapsed * 1000),
      feedback_condition = cond)
  }
  do.call(rbind, rows)
}
# Read every CSV in the two folders; fall back to the synthetic backup if both are empty.
d <- rbind(read_folder("data/no_feedback", "no_feedback"), read_folder("data/feedback", "feedback"))
if (is.null(d)) {
  d <- make_backup()
  DATA_SOURCE <- "backup"
  message("No CSV files in data/no_feedback or data/feedback: using SYNTHETIC BACKUP DATA ",
          "(group G0, six fictional people). These are not real measurements.")
} else {
  DATA_SOURCE <- "own"
  message("Read ", nrow(d), " rows for ", length(unique(d$participant)), " participant(s).")
}
# Derived columns used in the exercises
d$condition <- factor(d$feedback_condition, levels = c("no_feedback", "feedback"))
d$error_s   <- d$elapsed_seconds - d$target_seconds      # < 0: stopped too early
d$rel_error <- d$error_s / d$target_seconds             # -0.10 = 10 % too short
d$short     <- d$error_s < 0                              # TRUE if too short
d$order     <- (match(d$target_seconds, c(5, 10, 15)) - 1) * 10 + d$trial  # 1-30 within a session
head(d)

In [ ]:
library(tidyverse)
d |> count(participant, condition)

## B1. Pooled and individual views

We use the **no-feedback** trials and the relative error, so that 5, 10 and 15 s can be compared on one scale.

1. What does the pooled histogram hide that the boxplot by person shows?
2. Is the pooled mean relative error close to 0? Does that mean people are accurate?
3. Which is more affected by one unusually long trial: the mean or the median?

In [ ]:
nf <- d |> filter(condition == "no_feedback")
nf |> summarise(mean = mean(rel_error), median = median(rel_error), sd = sd(rel_error))

ggplot(nf, aes(x = rel_error)) +
  geom_histogram(bins = 30, fill = "grey70", colour = "white") +
  geom_vline(xintercept = 0, linetype = 2, linewidth = 1) +
  labs(title = paste("All no-feedback trials pooled:", DATA_SOURCE, "data"),
       x = "Relative error (-0.1 = 10 % too short)", y = "Number of trials")

In [ ]:
ggplot(nf, aes(x = participant, y = rel_error)) +
  geom_boxplot(fill = "grey85") +
  geom_hline(yintercept = 0, linetype = 2) +
  labs(title = "No-feedback trials by person", x = "Participant",
       y = "Relative error (-0.1 = 10 % too short)")

**Your answer:** write a few sentences here.

## B2. Who is consistently shorter, or longer, than the target?

For each person, count how many of the **30 no-feedback trials** were too short. We use a working rule:

| Too short in … of 30 trials | Label |
|---|---|
| 21–30 | consistently shorter |
| 10–20 | no clear difference |
| 0–9 | consistently longer |

Answer:

1. How many people in your group are consistently shorter, consistently longer, or show no clear difference?
2. Does a person’s direction depend on the target (5, 10 or 15 s)? Use the second table.
3. Suppose nobody had a bias, so every trial is a fair coin flip. What is the probability that one person gets 21 or more short trials? Out of 62 such people, about how many would get a “consistent” label by chance?
4. Why might 30 trials of one person **not** behave like 30 independent coin flips?

In [ ]:
counts <- nf |>
  group_by(participant) |>
  summarise(n_short = sum(short), n_trials = n()) |>
  mutate(label = cut(n_short, breaks = c(-1, 9, 20, 30),
                     labels = c("consistently longer", "no clear difference", "consistently shorter")))
counts
counts |> count(label, .drop = FALSE)

In [ ]:
# Too-short trials (out of 10) per person and target: one column per target
nf |>
  group_by(participant, target_seconds) |>
  summarise(n_short = sum(short), .groups = "drop") |>
  pivot_wider(names_from = target_seconds, values_from = n_short, names_prefix = "target_")

# Q3: chance of 21 or more short out of 30 if P(short) = 0.5
1 - pbinom(20, size = 30, prob = 0.5)

**Your answer:** write a few sentences here.

## B3. Does feedback improve accuracy, for everybody?

Accuracy here means the **mean absolute relative error**: on average, how far each trial is from the target, in percent. Smaller is better.

1. How many people in your group improved with feedback, and how many got worse? Did it help **everybody**?
2. If feedback did nothing, each person would improve with probability about 0.5. How surprising is your group’s count? (Edit the numbers in the cell.)
3. Did feedback mainly reduce people’s **bias** (signed error) or their **spread** (SD)?
4. **Pitfall:** the feedback session always came second. What else could explain an improvement? How could the experiment be designed to separate the explanations?

In [ ]:
# Mean absolute relative error per person and condition, one row per person
acc <- d |>
  group_by(participant, condition) |>
  summarise(mae = mean(abs(rel_error)), .groups = "drop") |>
  pivot_wider(names_from = condition, values_from = mae) |>
  mutate(change = feedback - no_feedback,   # negative = more accurate with feedback
         improved = change < 0)
acc
acc |> count(improved)

In [ ]:
# One line per person: left = no feedback, right = feedback
acc_long <- acc |>
  pivot_longer(c(no_feedback, feedback), names_to = "condition", values_to = "mae") |>
  mutate(condition = factor(condition, levels = c("no_feedback", "feedback")),
         result = ifelse(improved, "improved", "worse"))

ggplot(acc_long, aes(x = condition, y = mae, group = participant, colour = result)) +
  geom_line(linewidth = 1) +
  geom_point(size = 2) +
  geom_text(data = filter(acc_long, condition == "feedback"), aes(label = participant),
            hjust = -0.2, show.legend = FALSE) +
  scale_colour_manual(values = c(improved = "steelblue", worse = "firebrick")) +
  labs(title = "Accuracy without and with feedback", x = NULL,
       y = "Mean absolute relative error", colour = NULL)

In [ ]:
# Q2: chance of k or more improvements out of n people if feedback did nothing
n <- nrow(acc); k <- sum(acc$improved)
1 - pbinom(k - 1, size = n, prob = 0.5)

# Q3: bias (mean signed error) and spread (SD) by condition
d |>
  group_by(condition) |>
  summarise(mean = mean(rel_error), sd = sd(rel_error))

**Your answer:** write a few sentences here.

## B4. One plot of every person’s trials

Each panel is one person: blue circles without feedback, orange triangles with feedback. The grey lines separate the 5, 10 and 15 s sections.

Compare this with the pooled histogram in B1. Do you see any trend within a session, for example learning during the feedback trials?

In [ ]:
ggplot(d, aes(x = order, y = rel_error, colour = condition, shape = condition)) +
  geom_hline(yintercept = 0, linetype = 2) +
  geom_vline(xintercept = c(10.5, 20.5), colour = "grey70") +
  geom_point() +
  facet_wrap(~ participant, ncol = 3) +
  scale_colour_manual(values = c(no_feedback = "steelblue", feedback = "darkorange")) +
  labs(title = "Every trial, one panel per person",
       x = "Trial in session (1-10: 5 s, 11-20: 10 s, 21-30: 15 s)", y = "Relative error")

**Your answer:** write a few sentences here.

## B5. Predict your own next attempt

Choose yourself (or one person in the backup). Predict **one new no-feedback attempt at 10 seconds** by that person under the same protocol: give a central value and a plausible range, and say how you chose them.

1. Even if we knew this person’s mean exactly, why would the next attempt still be uncertain?
2. How would your prediction change for a **new** person who has not been measured?
3. Was your A2 prediction from this morning right?

This is an informal judgement, not a fitted Bayesian interval: we build those from tomorrow.

In [ ]:
me <- d$participant[1]   # change to your own ID, e.g. "G3P07"
d |>
  filter(participant == me, condition == "no_feedback", target_seconds == 10) |>
  summarise(n = n(), mean = mean(elapsed_seconds), sd = sd(elapsed_seconds),
            min = min(elapsed_seconds), max = max(elapsed_seconds))

**Your answer:** write a few sentences here.

## B6. Density versus probability

1. Let X be uniform on [0, 0.5]. What is its density? What is P(0.1 < X < 0.3)? What is P(X = 0.2)? Why can a density be larger than 1?
2. `dnorm(0, mean = 0, sd = 0.2)` returns 1.99. Is that a mistake? What is the probability that a timing error is **exactly** 0 s?

Hints: `dunif()` gives the density and `punif()` the CDF; an interval probability is a difference of two CDF values.

In [ ]:
# Your calculations here, e.g. dunif(0.2, min = 0, max = 0.5)

**Your answer:** write a few sentences here.

## B7. Your group’s question set

Question writers: write **three** questions on this afternoon’s topics, each with your own worked solution. At least one should come from something your group found hard. The checker re-runs any code and checks the solutions. The reporter notes your hardest open question for the plenary.

**Concept question** (what does something mean?):

**Our solution:**

**Computation question** (change a value and predict the effect):

**Our solution:**

**Pitfall question** (what could go wrong or be misread?):

**Our solution:**

**Hardest open question (reporter):**

---
**Everyone back in the room at 15:00 for Peer review B.** At 15:45, write your daily diary on **Mentimeter** (menti.com, code on the slide): 3 things you learned today and 3 things you are not yet familiar with.

## Optional extensions (only if you finish early)

**E1. The marble game.** A bag holds 4 marbles, each blue or white; five bags (0–4 blue) are equally likely. Drawing with replacement gives blue, white, blue. Compute the posterior probability of each bag. Then a fourth draw is blue: update again. What is the probability that the next draw is blue?

**E2. The casino die.** A die is fair with prior probability 0.95, or loaded with probability 0.05 (a six with probability 1/2, each other face 1/10). In 10 rolls you see 5 sixes. What is P(loaded | data)?

**E3.** Derive E[X] and Var(X) for X ~ Bernoulli(p), and check at p = 0.6 in R.

In [ ]:
# Optional work

## Completion and make-up work

Save your own copy of both notebooks with results and written answers. No homework is required if you finish the core in class.

If you miss Day 1, follow `day-1-makeup.md`: submit your own answers to the core of both notebooks (A1–A7 and B1–B7). You can collect your own data at home with the two webpages (they work offline); otherwise the notebooks use the synthetic backup. For the group questions in B2 and B3, answer for the backup group G0 or for your own data, and say which.